# 🔍 Analyse Exploratoire (EDA) & Données Manquantes
**Cours couverts :** Exploratory Data Analysis in Python, Dealing with Missing Data in Python

---
L'EDA est la première étape de tout projet de data science. Le but est de comprendre les données avant de modéliser.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
sns.set_theme(style="whitegrid")

# Créer un dataset réaliste avec des problèmes typiques
n = 500

df = pd.DataFrame({
    "age": np.random.normal(38, 12, n).astype(int),
    "salaire": np.random.lognormal(10.8, 0.5, n),
    "experience": np.random.randint(0, 35, n),
    "education": np.random.choice(["Bac", "Bac+2", "Bac+3", "Bac+5", "Doctorat"],
                                   n, p=[0.15, 0.2, 0.25, 0.3, 0.1]),
    "departement": np.random.choice(["IT", "Marketing", "RH", "Finance", "Ventes"], n),
    "genre": np.random.choice(["F", "M"], n),
    "satisfaction": np.random.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.1, 0.2, 0.35, 0.3]),
    "region": np.random.choice(["Île-de-France", "Auvergne-Rhône-Alpes", "Nouvelle-Aquitaine",
                                  "Occitanie", "PACA"], n)
})

# Ajout de données manquantes et d'outliers
idx_missing = np.random.choice(df.index, size=50, replace=False)
df.loc[idx_missing[:30], "salaire"] = np.nan
df.loc[idx_missing[30:45], "education"] = np.nan
df.loc[idx_missing[45:], "satisfaction"] = np.nan

# Quelques outliers dans l'âge
df.loc[np.random.choice(df.index, 5), "age"] = np.random.choice([5, 8, 120, 150], 5)

print("Dataset créé. Shape:", df.shape)
print("\nAperçu :")
df.head()

## 1. Vue d'ensemble du dataset

In [ ]:
print("=== INFORMATIONS GÉNÉRALES ===")
print(f"Lignes : {df.shape[0]}, Colonnes : {df.shape[1]}")
print(f"Mémoire : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
print("\n")

print("=== TYPES DE DONNÉES ===")
print(df.dtypes)
print(f"\nNb colonnes numériques  : {len(df.select_dtypes(include='number').columns)}")
print(f"Nb colonnes catégorielles: {len(df.select_dtypes(include='object').columns)}")

In [ ]:
print("=== STATISTIQUES DESCRIPTIVES ===")
df.describe()

In [ ]:
# Variables catégorielles
print("=== VARIABLES CATÉGORIELLES ===")
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col} ({df[col].nunique()} valeurs uniques):")
    print(df[col].value_counts())

## 2. Analyse des données manquantes

In [ ]:
print("=== DONNÉES MANQUANTES ===")
manquants = pd.DataFrame({
    "nb_manquants": df.isnull().sum(),
    "pct_manquants": (df.isnull().mean() * 100).round(1)
})
print(manquants[manquants["nb_manquants"] > 0])

In [ ]:
# Visualiser les patterns de données manquantes
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap des NaN
sns.heatmap(df.isnull(), cmap="YlOrRd", cbar=False, ax=axes[0], yticklabels=False)
axes[0].set_title("Carte des données manquantes (jaune = NaN)")

# Barres
manquants_non_zero = manquants[manquants["nb_manquants"] > 0]
axes[1].barh(manquants_non_zero.index, manquants_non_zero["pct_manquants"],
              color="coral", edgecolor="white")
axes[1].set_title("% de données manquantes par colonne")
axes[1].set_xlabel("% manquants")

plt.tight_layout()
plt.show()

In [ ]:
# Types de données manquantes :
# MCAR (Missing Completely At Random) : manque de façon aléatoire, pas de biais
# MAR  (Missing At Random) : le manque dépend d'autres variables observées
# MNAR (Missing Not At Random) : le manque dépend de la valeur manquante elle-même

# Tester si les NaN dans 'salaire' sont liés au département
df["salaire_manquant"] = df["salaire"].isnull().astype(int)
print("Proportion de salaires manquants par département :")
print(df.groupby("departement")["salaire_manquant"].mean().round(3))
# Si les proportions sont similaires → probablement MCAR
# Si elles diffèrent beaucoup → probablement MAR

## 3. Stratégies de traitement des données manquantes

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer

df_work = df.drop(columns=["salaire_manquant"]).copy()

print("Stratégies de traitement :")
print()

# 1. Suppression
df_drop = df_work.dropna()
print(f"1. Suppression complète : {len(df_work)} → {len(df_drop)} lignes ({len(df_work)-len(df_drop)} supprimées)")

# 2. Imputation par la médiane/mode (variables numériques)
df_median = df_work.copy()
median_sal = df_median["salaire"].median()
df_median["salaire"].fillna(median_sal, inplace=True)
print(f"2. Imputation médiane salaire : NaN → {median_sal:.0f}")

# 3. Imputation par groupe
df_group = df_work.copy()
df_group["salaire"] = df_group.groupby("departement")["salaire"].transform(
    lambda x: x.fillna(x.median())
)
print(f"3. Imputation par groupe (médiane par département)")

# 4. KNN Imputer
df_knn = df_work.copy()
numeriques = ["age", "salaire", "experience", "satisfaction"]
imputer_knn = KNNImputer(n_neighbors=5)
df_knn[numeriques] = imputer_knn.fit_transform(df_knn[numeriques])
print(f"4. KNN Imputer (k=5 voisins les plus proches)")
print(f"   NaN restants : {df_knn[numeriques].isnull().sum().sum()}")

In [ ]:
# Comparer l'impact des stratégies sur la distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df_work["salaire"].dropna().plot(kind="hist", bins=30, alpha=0.7, ax=axes[0], color="gray")
axes[0].set_title("Distribution originale (sans NaN)")

df_median["salaire"].plot(kind="hist", bins=30, alpha=0.7, ax=axes[1], color="steelblue")
axes[1].set_title("Imputation médiane globale\n(pic autour de la médiane)")

df_knn["salaire"].plot(kind="hist", bins=30, alpha=0.7, ax=axes[2], color="coral")
axes[2].set_title("KNN Imputer\n(distribution plus naturelle)")

plt.suptitle("Comparaison des stratégies d'imputation")
plt.tight_layout()
plt.show()

## 4. Détection et traitement des outliers

In [ ]:
# Méthodes de détection des outliers

# 1. Méthode IQR (valeurs aberrantes)
def detecter_outliers_iqr(serie):
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    borne_inf = Q1 - 1.5 * IQR
    borne_sup = Q3 + 1.5 * IQR
    return serie[(serie < borne_inf) | (serie > borne_sup)]

# 2. Méthode Z-score
def detecter_outliers_zscore(serie, seuil=3):
    z_scores = np.abs(stats.zscore(serie.dropna()))
    return serie[z_scores > seuil]

outliers_iqr = detecter_outliers_iqr(df["age"])
outliers_z = detecter_outliers_zscore(df["age"])

print("Outliers dans 'age' :")
print(f"  Méthode IQR    : {len(outliers_iqr)} outliers → valeurs: {sorted(outliers_iqr.values)}")
print(f"  Méthode Z-score: {len(outliers_z)} outliers → valeurs: {sorted(outliers_z.values)}")

In [ ]:
# Traitement des outliers
df_clean = df_work.copy()

# 1. Suppression (si clairement erronés)
df_clean = df_clean[df_clean["age"].between(18, 70)]

# 2. Winsorisation (cap aux percentiles)
p01 = df_clean["salaire"].quantile(0.01)
p99 = df_clean["salaire"].quantile(0.99)
df_clean["salaire_winsorised"] = df_clean["salaire"].clip(lower=p01, upper=p99)

# 3. Transformation logarithmique (pour données avec longue queue)
df_clean["log_salaire"] = np.log1p(df_clean["salaire"].fillna(df_clean["salaire"].median()))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df_clean["salaire"].dropna().hist(bins=40, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Distribution originale (salaire)")

df_clean["salaire_winsorised"].dropna().hist(bins=40, ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("Après Winsorisation (P1-P99)")

df_clean["log_salaire"].hist(bins=40, ax=axes[2], color="mediumseagreen", edgecolor="white")
axes[2].set_title("Transformation log\n(distribution plus symétrique)")

plt.tight_layout()
plt.show()

## 5. EDA Complète : analyse des relations

In [ ]:
df_eda = df_clean.dropna().copy()

# Encoder les catégorielles pour la corrélation
edu_ordre = {"Bac": 1, "Bac+2": 2, "Bac+3": 3, "Bac+5": 4, "Doctorat": 5}
df_eda["edu_num"] = df_eda["education"].map(edu_ordre)

# Matrice de corrélation
cols_corr = ["age", "salaire", "experience", "satisfaction", "edu_num"]
corr_matrix = df_eda[cols_corr].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Matrice de corrélation")
plt.tight_layout()
plt.show()

In [ ]:
# Analyse bivariée : effet de l'éducation sur le salaire
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
ordre_edu = ["Bac", "Bac+2", "Bac+3", "Bac+5", "Doctorat"]
sns.boxplot(data=df_eda, x="education", y="salaire", order=ordre_edu, ax=axes[0])
axes[0].set_title("Salaire selon le niveau d'éducation")
axes[0].tick_params(axis="x", rotation=45)

# Scatter avec régression
sns.regplot(data=df_eda, x="experience", y="salaire",
            scatter_kws={"alpha": 0.3}, line_kws={"color": "red"}, ax=axes[1])
axes[1].set_title("Expérience vs Salaire (avec droite de régression)")

plt.tight_layout()
plt.show()

In [ ]:
# Analyse 3 variables : age, salaire, département
g = sns.FacetGrid(df_eda, col="departement", col_wrap=3, height=3.5)
g.map(sns.scatterplot, "experience", "salaire", alpha=0.5)
g.set_titles("{col_name}")
g.set_axis_labels("Expérience", "Salaire")
plt.suptitle("Expérience vs Salaire par département", y=1.02)
plt.tight_layout()
plt.show()

## 6. Checklist EDA complète

In [ ]:
checklist = """
✅ CHECKLIST EDA :

1. COMPRENDRE LE CONTEXTE
   □ Quelle est la question métier ?
   □ Quelle est la variable cible (y) ?
   □ D'où viennent les données ?

2. INSPECTER LE DATASET
   □ df.shape, df.dtypes, df.head()
   □ df.describe() pour les numériques
   □ df[col].value_counts() pour les catégorielles

3. DONNÉES MANQUANTES
   □ df.isnull().sum()
   □ Visualiser le pattern (heatmap)
   □ Comprendre le type (MCAR/MAR/MNAR)
   □ Choisir la stratégie (suppression, imputation)

4. DISTRIBUTIONS
   □ Histogrammes pour chaque variable numérique
   □ Asymétrie, kurtosis, normalité
   □ Identifier les outliers (IQR ou Z-score)

5. RELATIONS
   □ Matrice de corrélation (numériques)
   □ Scatter plots pour les paires importantes
   □ Boxplots pour numérique vs catégorielle
   □ FacetGrid pour analyser par sous-groupe

6. INGÉNIERIE DES FEATURES (si pertinent)
   □ Créer de nouvelles variables (interactions, ratios)
   □ Transformer (log, sqrt pour distributions asymétriques)
   □ Encoder les catégorielles (ordinal, one-hot)
   □ Normaliser/Standardiser

7. RÉSUMÉ ET HYPOTHÈSES
   □ Quelles variables semblent prédictives ?
   □ Y a-t-il des problèmes de qualité des données ?
   □ Quelles features à créer ?
"""
print(checklist)

In [ ]:
# Encodage des variables catégorielles (nécessaire pour ML)
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

print("=== TECHNIQUES D'ENCODAGE ===")
print()

# 1. Ordinal Encoding (pour variables avec ordre)
edu_order = [["Bac", "Bac+2", "Bac+3", "Bac+5", "Doctorat"]]
oe = OrdinalEncoder(categories=edu_order)
df_eda["education_encoded"] = oe.fit_transform(df_eda[["education"]])
print("1. Ordinal Encoding (éducation) :")
print(df_eda[["education", "education_encoded"]].drop_duplicates().sort_values("education_encoded"))

# 2. One-Hot Encoding (pour variables sans ordre)
dept_dummies = pd.get_dummies(df_eda["departement"], prefix="dept", drop_first=True)
print("\n2. One-Hot Encoding (département) :")
print(dept_dummies.head(5))